In [ ]:
### CATS
### ETTH
### 3 epocas aproximadaemtne 10 segs
### 10 epocas aproximadamente 30 segs
### M4
## No termina smh
#Epoch: 1 cost time: 259.2893192768097
#iters: 17300, epoch: 10 | loss: 0.1217130

### YEARLY

### SMAPE 13.263
### MASE  2.967
### OWA   0.779



import torch
import pandas as pd, numpy as np, glob

import os, shutil
if os.path.exists('/content/CATS-main'):
    shutil.rmtree('/content/CATS-main')
!unzip -q /content/CATS-main.zip -d /content/
%cd /content/CATS-main


!pip install -q matplotlib pandas scikit-learn

!grep -rl "np.Inf" --include="*.py" . | xargs sed -i 's/np\.Inf/np.inf/g' 2>/dev/null
!grep -rl "np.NaN" --include="*.py" . | xargs sed -i 's/np\.NaN/np.nan/g' 2>/dev/null

!mkdir -p ./dataset/m4
#!unzip -q /content/Dataset.zip -d /tmp/m4raw
!cp /content/Yearly-Train.csv ./dataset/m4/
!cp /content/Yearly-Test.csv ./dataset/m4/

df_tr = pd.read_csv('./dataset/m4/Yearly-Train.csv', header=0, index_col=0)
df_te = pd.read_csv('./dataset/m4/Yearly-Test.csv',  header=0, index_col=0)
L, H = 12, 6

rows = []

for sid in df_tr.index:
    tr = df_tr.loc[sid].dropna().values
    te = df_te.loc[sid].values if sid in df_te.index else None
    if len(tr) >= L and te is not None:
        rows.extend(tr[-L:].tolist())
        rows.extend(te[:H].tolist())
df_flat = pd.DataFrame({
    'date': pd.date_range('2000-01-01', periods=len(rows), freq='h'),
    'OT': rows
})
df_flat.to_csv('./dataset/m4/Yearly_flat.csv', index=False)

print()
print("Test M4")

!python -u run.py \
  --is_training 1 \
  --root_path ./dataset/m4/ \
  --data_path Yearly_flat.csv \
  --model_id m4_Yearly \
  --model CATS \
  --data custom \
  --features S \
  --seq_len 12 \
  --pred_len 6 \
  --label_len 0 \
  --d_layers 3 \
  --dec_in 1 \
  --des 'M4_Yearly' \
  --itr 1 \
  --d_model 64 \
  --d_ff 256 \
  --n_heads 8 \
  --QAM_end 0.2 \
  --batch_size 32 \
  --patch_len 6 \
  --stride 6 \
  --train_epochs 30 \
  --patience 10 \
  --inverse 1 \
  --num_workers 8


Se han truncado las últimas 5000 líneas del flujo de salida.
	speed: 0.0140s/iter; left time: 3450.3682s
	iters: 6700, epoch: 3 | loss: 0.5073406
	speed: 0.0139s/iter; left time: 3424.1645s
	iters: 6800, epoch: 3 | loss: 0.2574264
	speed: 0.0137s/iter; left time: 3388.8149s
	iters: 6900, epoch: 3 | loss: 0.7117779
	speed: 0.0161s/iter; left time: 3965.2447s
	iters: 7000, epoch: 3 | loss: 0.2785980
	speed: 0.0184s/iter; left time: 4545.4804s
	iters: 7100, epoch: 3 | loss: 0.5292332
	speed: 0.0138s/iter; left time: 3399.0388s
	iters: 7200, epoch: 3 | loss: 0.4125741
	speed: 0.0138s/iter; left time: 3407.1935s
	iters: 7300, epoch: 3 | loss: 0.3978314
	speed: 0.0140s/iter; left time: 3435.8750s
	iters: 7400, epoch: 3 | loss: 0.6216545
	speed: 0.0140s/iter; left time: 3437.3130s
	iters: 7500, epoch: 3 | loss: 0.8128474
	speed: 0.0138s/iter; left time: 3396.8825s
	iters: 7600, epoch: 3 | loss: 0.3089870
	speed: 0.0140s/iter; left time: 3433.7431s
	iters: 7700, epoch: 3 | loss: 0.2622979
	spe

In [ ]:
results_dir = sorted(glob.glob('./results/m4_Yearly_*/'))[-1]

preds = np.load(results_dir + 'pred.npy')
trues = np.load(results_dir + 'true.npy')
preds = preds.squeeze(-1)
trues = trues.squeeze(-1)

df_tr = pd.read_csv('./dataset/m4/Yearly-Train.csv', header=0, index_col=0)
df_te = pd.read_csv('./dataset/m4/Yearly-Test.csv',  header=0, index_col=0)

L, H, SP = 12, 6, 1
NAIVE2_SMAPE = 16.34
NAIVE2_MASE  = 3.974


### FORMULAS

def smape(a, f):
    a, f = np.array(a), np.array(f)
    d = np.abs(a) + np.abs(f)
    return 200 * np.mean(np.abs(a - f)[d != 0] / d[d != 0])

def mase(a, f, insample, sp):
    scale = np.mean(np.abs(insample[sp:] - insample[:-sp])) + 1e-8
    return np.mean(np.abs(np.array(a) - np.array(f))) / scale

smapes, mases = [], []
valid_sids = [sid for sid in df_tr.index
              if len(df_tr.loc[sid].dropna()) >= L and sid in df_te.index]

for i, sid in enumerate(valid_sids[:len(preds)]):
    insample = df_tr.loc[sid].dropna().values.astype(float)
    actual   = df_te.loc[sid].values.astype(float)[:H]
    forecast = preds[i]
    smapes.append(smape(actual, forecast))
    mases.append(mase(actual, forecast, insample, SP))

mean_smape = np.nanmean(smapes)
mean_mase  = np.nanmean(mases)
mean_owa   = 0.5 * (mean_smape / NAIVE2_SMAPE + mean_mase / NAIVE2_MASE)

print("SMAPE: ", mean_smape)
print("MASE: ", mean_mase)
print("OWA: ", mean_owa)



SMAPE:  74.85677767517488
MASE:  30.580294399246807
OWA:  6.138144919985088


2.3 mins por epoca

54 mins hora por 20 epocas

In [ ]:
### Fredformer
### ETTH
### 3 epocas 30 segs aprox
### 10 epocas 1 min aprox
### M4
###
###

import torch, os, shutil, time

if os.path.exists('/content/Fredformer-main'):
    shutil.rmtree('/content/Fredformer-main')
!unzip -q /content/Fredformer-main.zip -d /content/
%cd /content/Fredformer-main

!pip install -q einops reformer-pytorch sktime torchinfo thop

!grep -rl "np\.Inf" --include="*.py" . | xargs -r sed -i 's/np\.Inf/np.inf/g'
!grep -rl "np\.NaN" --include="*.py" . | xargs -r sed -i 's/np\.NaN/np.nan/g'


!mkdir -p ./dataset
#!wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv -O ./dataset/ETTh1.csv
#!wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh2.csv -O ./dataset/ETTh2.csv
#!wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv -O ./dataset/ETTm1.csv
#!wget -q https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm2.csv -O ./dataset/ETTm2.csv

from huggingface_hub import hf_hub_download
!mkdir -p ./dataset/m4
for freq in ['Monthly', 'Quarterly', 'Yearly', 'Weekly', 'Daily', 'Hourly']:
    for split in ['Train', 'Test']:
        hf_hub_download(
            repo_id='thuml/Time-Series-Library',
            filename=f'm4/{freq}-{split}.csv',
            repo_type='dataset',
            local_dir='./dataset'
        )

for pred_len in [96]:
    print()
    print("Test")

    !python -u run_longExp.py \
      --random_seed 2021 \
      --is_training 1 \
      --root_path ./dataset/ \
      --data_path ETTh1.csv \
      --model_id ETTh1_96_{pred_len} \
      --model Fredformer \
      --data ETTh1 \
      --features M \
      --seq_len 96 \
      --pred_len {pred_len} \
      --enc_in 7 \
      --d_model 24 \
      --d_ff 128 \
      --dropout 0.3 \
      --fc_dropout 0.3 \
      --patch_len 4 \
      --stride 4 \
      --des 'Test Fredformer' \
      --train_epochs 10 \
      --patience 10 \
      --itr 1 \
      --batch_size 128 \
      --learning_rate 0.0001 \
      --cf_dim 128 \
      --cf_depth 2 \
      --cf_heads 8 \
      --cf_mlp 96 \
      --cf_head_dim 32 \
      --use_nys 0 \
      --individual 0 \
      --num_workers 2



/content/Fredformer-main
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 10.5 MB/s eta 0:00:00

Test
Args in experiment:
Namespace(cf_dim=128, cf_drop=0.2, cf_depth=2, cf_heads=8, cf_mlp=96, cf_head_dim=32, cf_weight_decay=0, cf_p=1, use_nys=0, mlp_drop=0.3, ablation=0, random_seed=2021, is_training=1, model_id='ETTh1_96_96', model='Fredformer', data='ETTh1', root_path='./dataset/', data_path='ETTh1.csv', features='M', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=48, pred_len=96, fc_dropout=0.3, head_dropout=0.0, patch_len=4, stride=4, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, embed_type=0, enc_in=7, dec_in=7, c_out=7, d_model=24, mlp_hidden=64, n_heads=8, e_layers=2, d_layers=1, d_ff=128, moving_avg=25, factor=1, distil=True, dropout=0.3, embed='timeF', activation='gelu', output_attention=False, do_predict

Para 10 epocas (GPU 4)

CATS 10 segs

test 2785
mse:0.371696412563324, mae:0.39527738094329834, rmse:0.6096690893173218

Fredformer 59 segs

test 2785
mse:0.3760049641132355, mae:0.39402705430984497, rse:0.5814498066902161





CATS: Mejor MSE, Entrenamiento veloz

Fredformer: Mejor MAE,